In [24]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [3]:
words = open("names.txt", "r").read().splitlines()

In [4]:
len(words)

32033

In [5]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [6]:
chars = sorted(list(set(''.join(words)))) 
chars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [7]:
# Build the mappings
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {ch: i for i, ch in stoi.items()}

itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [33]:
# Build the dataset
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]


In [34]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [35]:
X.shape, Y.shape, X.dtype, Y.dtype

(torch.Size([228146, 3]), torch.Size([228146]), torch.int64, torch.int64)

In [36]:
gen = torch.Generator().manual_seed(123)

In [37]:
C = torch.randn((27, 2), generator=gen)

In [38]:
C[5]

tensor([ 0.6984, -1.4097])

In [39]:
C[X].shape

torch.Size([228146, 3, 2])

In [40]:
X[1]

tensor([0, 0, 5])

In [41]:
C[X[1]]

tensor([[ 0.3374, -0.1778],
        [ 0.3374, -0.1778],
        [ 0.6984, -1.4097]])

In [42]:
# Embedding
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [43]:
# Define the parameters
W1 = torch.randn(6, 100, generator=gen)
b1 = torch.randn(100, generator=gen)
W2 = torch.randn(100, 27, generator=gen)
b2 = torch.randn(27, generator=gen)
parameters = [C, W1, b1, W2, b2]


In [44]:
for p in parameters:
    p.requires_grad = True

In [45]:
# Training loop
epochs = 100

for epoch in range(epochs):
    # construct mini batches
    batch_size = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[batch_size]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[batch_size])
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item()}")

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad

Epoch 0: Loss = 16.417516708374023
Epoch 10: Loss = 11.20736312866211
Epoch 20: Loss = 8.367496490478516
Epoch 30: Loss = 5.4274678230285645
Epoch 40: Loss = 3.944939613342285
Epoch 50: Loss = 5.065392971038818
Epoch 60: Loss = 3.416679620742798
Epoch 70: Loss = 3.7195208072662354
Epoch 80: Loss = 2.784365177154541
Epoch 90: Loss = 3.357410430908203


In [47]:
# Overall loss calculation
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)

loss.item()

3.3840246200561523